# MacroRegimes3 — Part 1: Data Collection & Cleaning

This notebook builds the monthly macro dataset used by the three regime classifiers
(`TreeClassifier.ipynb`, `PVRFClassifier.ipynb`, `HMMClassifier.ipynb`). It is the only
notebook that touches the internet. Split out of `MacroRegimes2.ipynb`.

**Sources:** FRED (`fredapi`), yfinance, and supplementary files (`BAMLH0A0HYM2.csv`, `gold.csv`,
`fear-greed_sentiment.xls`, `data_gpr_export.xls`, `feds200628.csv`). See Section 1 for the variable list.

**Archive policy (important).** Raw monthly data is kept in `MacroRegimes3_MacroData_Archive.csv`.
Every run downloads fresh data and *merges it into* the archive:
- new dates and new series are **added**,
- values already in the archive are **never overwritten silently**,
- if a fresh value disagrees with an archived value (e.g. a FRED revision) the conflicts are
  listed and you are asked whether to overwrite.

This matters because some FRED series (e.g. `BAMLH0A0HYM2`) only serve a trailing 3-year window
for licensing reasons: the archive is what lets a run in 2030 still see 2026. Answering "n" to the
overwrite prompt also makes the archive a crude first-vintage record of each series.

**Outputs (all in `data_folder`, i.e. `DataAndResults/`):**

| File | Contents | Used by |
|---|---|---|
| `MacroRegimes3_MacroData_Archive.csv` | raw monthly levels, all series, full history | this notebook (next run) |
| `MacroRegimes3_Data_Levels.csv` | cleaned levels incl. derived series, from `start_date` | Tree, HMM (plots, stats) |
| `MacroRegimes3_Data_Quantiles.csv` | rolling 5-year quantile ranks, aligned to the target | Tree, PV-RF |
| `MacroRegimes3_Data_Target.csv` | Svensson zero-coupon yields (13 maturities) + NSS parameters | Tree, PV-RF, HMM |
| `MacroRegimes3_Data_Recession.csv` | NBER recession indicator (`USREC`) | Tree, PV-RF, HMM |
| `MacroRegimes3_Data_Columns.csv` | variable → group / proxy-variable category / transform | Tree, PV-RF, HMM |

## 0. Imports and User Inputs

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from fredapi import Fred
except ImportError:
    !pip install fredapi -q
    from fredapi import Fred

try:
    import yfinance as yf
except ImportError:
    !pip install yfinance -q
    import yfinance as yf

In [ ]:
# Folder holding every CSV/Excel input and every output of the four notebooks.
# On Colab the Drive folder is mounted; locally the notebooks sit in MacroRegimes3/ next to DataAndResults/.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    data_folder = '/content/drive/MyDrive/DataAndResults'
except ImportError:
    data_folder = os.path.join('..', 'DataAndResults')

os.makedirs(data_folder, exist_ok=True)
print(f'Data folder ready: {data_folder}')

def find_file(name):
    """Return the path of a supplementary file in the data folder, or None if it is missing."""
    path = os.path.join(data_folder, name)
    return path if os.path.exists(path) else None

In [ ]:
# FRED series (USREC is the NBER recession indicator; it is stored but never used as a feature)
fred = Fred(api_key='BRUH')
fred_series = [
    'NEWORDER', 'INDPRO', 'CUMFNS', 'BOPGTB', 'ISRATIO', 'RPI', 'DPCERA3M086SBEA', 'PCEPILFE', 'RSXFS', 'JTSJOL', 'ICSA', 'CCSA', 'AHETPI', 'MORTGAGE30US', 'PERMIT', 'COMPUTSA', 'BUSLOANS', 'M2SL', 'FEDFUNDS', 'CPIAUCSL', 'PPIACO', 'MICH', 'EXPINF1YR',
    'TB3MS', 'TB6MS', 'GS1', 'GS2', 'GS5', 'GS10', 'BAMLH0A0HYM2', 'BAMLC0A0CM', 'T10Y2YM', 'T10Y3MM',
    'TWEXBMTH', 'TWEXBGSMTH', 'EXSZUS', 'EXJPUS', 'EXUSUK', 'EXUSEU', 'EXBEUS', 'EXGEUS', 'EXDNUS', 'EXFRUS', 'EXUSIR', 'EXITUS', 'EXNEUS', 'CCUSMA02LUM618N', 'EXSPUS', 'EXGRUS', 'EXPOUS', 'EXCAUS',
    'WPU012202', 'WPU0121', 'WTISPLC', 'WPUSI019011', 'PNGASUSUSDM',
    'USACSCICP02STSAM', 'UMCSENT', 'KCFSI', 'STLFSI4', 'NFCI', 'USEPUINDXD', 'EPUMONETARY', 'EPUTRADE', 'VIXCLS',
    'USREC'
]

# yfinance series
yf_series = ['^GSPC', '^IXIC', '^DJI', '^N225', '^FCHI', '^AEX', '^BFX', '^GSPTSE', '^HSI', '^FTSE', '^GDAXI']

# Data start and end dates for the modelling sample
start_date = '1990-01-01'
end_date = '2100-01-01'

# Rolling quantile standardization window (months). Paper uses 120; we use 60 to preserve more data.
rolling_window = 60

# Archive settings
archive_file = f'{data_folder}/MacroRegimes3_MacroData_Archive.csv'
conflict_tol = 1e-6  # relative difference above which an archived vs fresh value counts as a conflict

# Output file names
out_levels    = f'{data_folder}/MacroRegimes3_Data_Levels.csv'
out_quantiles = f'{data_folder}/MacroRegimes3_Data_Quantiles.csv'
out_target    = f'{data_folder}/MacroRegimes3_Data_Target.csv'
out_recession = f'{data_folder}/MacroRegimes3_Data_Recession.csv'
out_columns   = f'{data_folder}/MacroRegimes3_Data_Columns.csv'

## 1. Variable List

Based on Zhao et al. (2025). Every series is stored at its **raw level** in the archive; ratios,
inverses and bridged series are computed later (Section 4) so they can always be rebuilt.

**Macroeconomic Factor** — Production: NEWORDER, INDPRO, CUMFNS, BOPGTB, ISRATIO · Consumer: RPI, PCEPILFE,
DPCERA3M086SBEA, RSXFS · Labour: JTSJOL, ICSA, CCSA/ICSA, AHETPI (UNRATE/PAYEMS omitted: lag and heavy revisions) ·
Housing: MORTGAGE30US, PERMIT, COMPUTSA/PERMIT · Liquidity & Credit: BUSLOANS, M2SL, FEDFUNDS ·
Inflation: CPIAUCSL, PPIACO, MICH, EXPINF1YR

**Bond Market** — TB3MS, TB6MS, GS1, GS2, GS5, GS10, BAMLH0A0HYM2 (CSV for pre-2023), BAMLC0A0CM (CSV for pre-2023), T10Y2YM, T10Y3MM

**Foreign Exchange** — Trade-weighted dollar (TWEXBGSMTH extended back with TWEXBMTH), 1/EXSZUS, 1/EXJPUS, EXUSUK,
ECU/EUR (fixed 1989 ECU basket pre-1999, EXUSEU post-1999), 1/EXCAUS

**Commodities** — WPU012202 (corn), WPU0121 (wheat), LBMA gold (`gold.csv`), WTISPLC, WPUSI019011 (copper), PNGASUSUSDM

**Stock Market** — ^GSPC, ^IXIC, ^DJI, ^N225, ^FCHI, ^AEX, ^BFX, ^GSPTSE, ^HSI, ^FTSE, ^GDAXI

**Sentiment** — USACSCICP02STSAM, UMCSENT, AAII bullish/bearish (`fear-greed_sentiment.xls`), KCFSI, STLFSI4, NFCI

**Uncertainty** — GPR/GPRT/GPRA (`data_gpr_export.xls`), USEPUINDXD, EPUMONETARY, EPUTRADE, VIXCLS

## 2. Download Fresh Data

Everything is resampled to month-start (`MS`) labels. FRED daily/weekly series use the monthly **mean**,
equity indices use the month-end **close**, exactly as in MacroRegimes2.

> TODO (known issue, not fixed yet): mixing monthly-mean, month-end and (in Section 7) first-of-month
> conventions means a row dated e.g. 2015-03-01 contains information from different points in March.

The current, incomplete month is dropped so that partial-month averages are never archived.

In [ ]:
# 2.1 FRED
print('Fetching FRED data...')
fred_frames = []
for ticker in fred_series:
    try:
        s = fred.get_series(ticker, observation_start='1900-01-01')
        s.name = ticker
        fred_frames.append(s)
        print(f'  fetched {ticker}')
    except Exception as e:
        print(f'  ERROR {ticker}: {e}')

if fred_frames:
    fresh_fred = pd.concat(fred_frames, axis=1)
    fresh_fred.index = pd.to_datetime(fresh_fred.index)
    fresh_fred = fresh_fred.sort_index().resample('MS').mean()
else:
    fresh_fred = pd.DataFrame()
print(f'FRED: {fresh_fred.shape[1]} series')

In [ ]:
# 2.2 Supplementary CSVs for the ICE BofA spreads (FRED only serves the trailing 3 years)
for ticker in ['BAMLH0A0HYM2', 'BAMLC0A0CM']:
    path = find_file(f'{ticker}.csv')
    if path is None:
        print(f'{ticker}.csv not found. Skipping.')
        continue
    csv = pd.read_csv(path, index_col='observation_date', parse_dates=True).ffill().resample('MS').mean()
    if ticker in fresh_fred.columns:
        fresh_fred[ticker] = csv[ticker].combine_first(fresh_fred[ticker])
    else:
        fresh_fred[ticker] = csv[ticker]
    print(f'{ticker}: merged {path}')

In [ ]:
# 2.3 yfinance (month-end close)
print('Fetching yfinance data...')
try:
    yf_raw = yf.download(tickers=yf_series, start='1900-01-01', end=end_date, auto_adjust=True, progress=False)
    fresh_yf = yf_raw['Close'].resample('MS').last()
    print(f'yfinance: {fresh_yf.shape[1]} series')
except Exception as e:
    print(f'ERROR yfinance: {e}')
    fresh_yf = pd.DataFrame()

In [ ]:
# 2.4 Supplementary Excel/CSV files (gold, AAII sentiment, geopolitical risk)
fresh_files = pd.DataFrame()

path = find_file('gold.csv')
if path:
    gold = pd.read_csv(path, index_col='d', parse_dates=True).ffill().resample('MS').mean()
    fresh_files['GOLD'] = gold['v/0']
    print('LBMA gold: loaded')
else:
    print('gold.csv not found. Skipping.')

path = find_file('fear-greed_sentiment.xls')
if path:
    aaii = pd.read_excel(path, sheet_name='SENTIMENT', skiprows=5, header=None)
    aaii.columns = ['Date', 'Bullish', 'Neutral', 'Bearish'] + list(aaii.columns[4:])
    aaii['Date'] = pd.to_datetime(aaii['Date'], errors='coerce')
    aaii = aaii.dropna(subset=['Date', 'Bullish', 'Bearish']).set_index('Date').sort_index()
    aaii = aaii[['Bullish', 'Bearish']].resample('MS').mean()
    fresh_files['AAII_Bullish'] = aaii['Bullish']
    fresh_files['AAII_Bearish'] = aaii['Bearish']
    print('AAII sentiment: loaded')
else:
    print('fear-greed_sentiment.xls not found. Skipping.')

path = find_file('data_gpr_export.xls')
if path:
    gpr = pd.read_excel(path, sheet_name='Sheet1')
    gpr = gpr.dropna(subset=['month', 'GPR', 'GPRT', 'GPRA'])
    gpr['month'] = pd.to_datetime(gpr['month'], errors='coerce')
    gpr = gpr.set_index('month').sort_index()[['GPR', 'GPRT', 'GPRA']].resample('MS').mean()
    for col in ['GPR', 'GPRT', 'GPRA']:
        fresh_files[col] = gpr[col]
    print('Geopolitical risk: loaded')
else:
    print('data_gpr_export.xls not found. Skipping.')

In [ ]:
# 2.5 Combine all fresh sources and drop the current (incomplete) month
sources = [df for df in [fresh_fred, fresh_yf, fresh_files] if not df.empty]
fresh = pd.concat(sources, axis=1, sort=True) if sources else pd.DataFrame(index=pd.DatetimeIndex([]))
fresh.index.name = 'date'
this_month = pd.Timestamp.today().to_period('M').to_timestamp()
fresh = fresh[fresh.index < this_month]
print(f'Fresh data: {fresh.shape[0]} months × {fresh.shape[1]} series')

## 3. Merge Into the Archive

Rules: archived values win by default; new dates/series are added; conflicts are reported and you decide.
On the very first run the archive is seeded from `fred_raw.csv` (the raw FRED pull saved by MacroRegimes2) if it is in the data folder.

In [ ]:
if os.path.exists(archive_file):
    archive = pd.read_csv(archive_file, index_col='date', parse_dates=True)
    print(f'Loaded archive: {archive.shape[0]} months × {archive.shape[1]} series')
else:
    legacy = find_file('fred_raw.csv')
    if legacy:
        archive = pd.read_csv(legacy, index_col=0, parse_dates=True)
        archive = archive[[c for c in archive.columns if c in fred_series]]  # raw tickers only
        archive.index.name = 'date'
        print(f'No archive yet. Seeded from {legacy}: {archive.shape[0]} months × {archive.shape[1]} series')
    else:
        archive = pd.DataFrame(index=pd.DatetimeIndex([], name='date'))
        print('No archive yet. Starting from scratch.')

In [ ]:
def merge_into_archive(archive, fresh, tol):
    """
    Add fresh data to the archive without overwriting.
    Returns the merged table (archive values kept where both exist) and a table of conflicts.
    """
    idx = archive.index.union(fresh.index)
    cols = list(archive.columns) + [c for c in fresh.columns if c not in archive.columns]
    old = archive.reindex(index=idx, columns=cols)
    new = fresh.reindex(index=idx, columns=cols)

    both = old.notna() & new.notna()
    differs = (old - new).abs() > tol * (1 + old.abs())
    conflict_mask = both & differs

    conflicts = []
    for col in cols:
        dates = idx[conflict_mask[col].values]
        for d in dates:
            conflicts.append({'date': d, 'series': col, 'archived': old.at[d, col], 'fresh': new.at[d, col]})
    conflicts = pd.DataFrame(conflicts, columns=['date', 'series', 'archived', 'fresh'])
    conflicts['abs_diff'] = (conflicts['fresh'] - conflicts['archived']).abs()

    merged = old.combine_first(new)  # archive wins, gaps filled from fresh
    return merged, conflicts, new

merged, conflicts, fresh_aligned = merge_into_archive(archive, fresh, conflict_tol)

n_new_cells = (merged.notna() & archive.reindex_like(merged).isna()).sum().sum()
print(f'New values added: {n_new_cells}')
print(f'Conflicts (archived value ≠ fresh value): {len(conflicts)}')

In [ ]:
if len(conflicts) > 0:
    summary = conflicts.groupby('series').agg(
        n_conflicts=('date', 'size'),
        first_date=('date', 'min'),
        last_date=('date', 'max'),
        max_abs_diff=('abs_diff', 'max'),
    )
    print('Conflicts by series:')
    print(summary.to_string())
    print('\nFirst 20 conflicts:')
    print(conflicts.head(20).to_string(index=False))

    answer = input("\nOverwrite archived values with fresh values? 'y' = all, 'n' = none, "
                   "or a comma-separated list of series names: ").strip()
    if answer.lower() == 'y':
        overwrite_cols = list(summary.index)
    elif answer.lower() == 'n':
        overwrite_cols = []
    else:
        overwrite_cols = [c.strip() for c in answer.split(',') if c.strip() in summary.index]

    for col in overwrite_cols:
        rows = conflicts.loc[conflicts['series'] == col, 'date']
        merged.loc[rows, col] = fresh_aligned.loc[rows, col]
    print(f'Overwrote {len(overwrite_cols)} series: {overwrite_cols}')

archive = merged.sort_index()
archive.to_csv(archive_file, index=True)
print(f'\nArchive saved: {archive.shape[0]} months × {archive.shape[1]} series → {archive_file}')

## 4. Derived Series
Ratios, inverses, the trade-weighted dollar bridge and the ECU/EUR bridge, computed from the archive.

In [ ]:
raw = archive.copy()

# 4.1 Ratios and inverses
raw['CCSA/ICSA'] = raw['CCSA'] / raw['ICSA']
raw['COMPUTSA/PERMIT'] = raw['COMPUTSA'] / raw['PERMIT']
raw['1/EXSZUS'] = 1 / raw['EXSZUS']
raw['1/EXJPUS'] = 1 / raw['EXJPUS']
raw['1/EXCAUS'] = 1 / raw['EXCAUS']

In [ ]:
# 4.2 Extend the modern trade-weighted dollar (TWEXBGSMTH) backwards with the legacy index (TWEXBMTH)
def sigmoid_bridge(df, legacy_col='TWEXBMTH', modern_col='TWEXBGSMTH', k=0.5):
    """Blend two overlapping index series with a sigmoid weight centred on the middle of the overlap."""
    overlap = df[df[legacy_col].notna() & df[modern_col].notna()].index
    if len(overlap) == 0:
        raise ValueError('No overlapping data found between the two series.')
    transition_date = overlap[len(overlap) // 2]

    months_from_center = (df.index.year - transition_date.year) * 12 + (df.index.month - transition_date.month)
    weight_modern = 1 / (1 + np.exp(-k * months_from_center))
    weight_legacy = 1 - weight_modern

    scale_factor = df.loc[transition_date, modern_col] / df.loc[transition_date, legacy_col]
    legacy_scaled = df[legacy_col] * scale_factor

    combined = legacy_scaled.fillna(0) * weight_legacy + df[modern_col].fillna(0) * weight_modern
    combined = combined[df[legacy_col].notna() | df[modern_col].notna()]
    return combined, legacy_scaled, transition_date

try:
    combined_series, legacy_scaled, t_center = sigmoid_bridge(raw)
    raw['Extended_TWEXBGSMTH'] = combined_series
    raw['Inverted_TWEXBGSMTH'] = 1 / raw['Extended_TWEXBGSMTH']

    plt.figure(figsize=(12, 4))
    plt.plot(raw.index, legacy_scaled, label='Legacy (TWEXBMTH) scaled', alpha=0.4, lw=2)
    plt.plot(raw.index, raw['TWEXBGSMTH'], label='Modern (TWEXBGSMTH)', alpha=0.4, lw=2)
    plt.plot(raw.index, raw['Extended_TWEXBGSMTH'], label='Extended', color='black', lw=0.8, ls='--')
    plt.axvline(t_center, color='red', ls=':', label=f'Sigmoid centre ({t_center.date()})')
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
except Exception as e:
    print(f'Error merging TWEXBMTH and TWEXBGSMTH: {e}. Skipping.')

In [ ]:
# 4.3 ECU (fixed 1989 basket) before 1999, euro after 1999
# TODO (known issue, not fixed yet): the reconstructed ECU is ~13% above the euro at the 1999 parity
# check, so a basket weight or unit is wrong. Harmless today because the sample starts in 2006 and
# the YoY transform absorbs a constant rescale, but fix before extending the sample back.
def ecu_euro_series(df):
    basket = {'EXBEUS': 3.301, 'EXGEUS': 0.6242, 'EXDNUS': 0.1976, 'EXFRUS': 1.332, 'EXUSUK': 0.08784,
              'EXUSIR': 0.008552, 'EXITUS': 151.8, 'EXNEUS': 0.2198, 'CCUSMA02LUM618N': 0.13,
              'EXSPUS': 6.885, 'EXGRUS': 1.44, 'EXPOUS': 1.393}
    terms = []
    for code, amount in basket.items():
        if code in df.columns:
            term = amount * df[code] if code in ['EXUSUK', 'EXUSIR'] else amount / df[code]  # USD-per-unit vs units-per-USD
            terms.append(term.ffill().fillna(0))
    ecu = pd.concat(terms, axis=1).sum(axis=1)
    ecu[ecu.index < pd.Timestamp('1989-09-21')] = np.nan  # basket weights fixed from Sept 1989

    transition = pd.Timestamp('1999-01-01')
    combined = pd.Series(index=df.index, dtype='float64')
    combined[df.index <= transition] = ecu
    combined[df.index > transition] = df['EXUSEU']

    if transition in df.index:
        ecu_val, euro_val = ecu.loc[transition], df.loc[transition, 'EXUSEU']
        print(f'ECU at transition: {ecu_val:.4f} | Euro: {euro_val:.4f} | difference: {ecu_val - euro_val:.4f}')
        combined[df.index <= transition] *= euro_val / ecu_val  # force continuity at the transition
    return combined

try:
    raw['ECU_EUR_USD'] = ecu_euro_series(raw)
    plt.figure(figsize=(12, 4))
    plt.plot(raw.loc['1995':'2002', 'ECU_EUR_USD'], lw=2, label='ECU/EUR (USD value)')
    plt.axvline(pd.Timestamp('1999-01-01'), color='red', ls='--', alpha=0.6, label='Transition')
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
except Exception as e:
    print(f'Error combining ECU/Euro series: {e}. Skipping.')

In [ ]:
# 4.4 Drop the raw components that are only used to build derived series
drop_cols = ['CCSA', 'COMPUTSA', 'TWEXBGSMTH', 'TWEXBMTH',
             'EXUSEU', 'EXBEUS', 'EXGEUS', 'EXDNUS', 'EXFRUS', 'EXUSIR', 'EXITUS', 'EXNEUS', 'CCUSMA02LUM618N', 'EXSPUS', 'EXGRUS', 'EXPOUS',
             'EXSZUS', 'EXJPUS', 'EXCAUS']
raw = raw.drop(columns=[c for c in drop_cols if c in raw.columns])

print('Amount of data:')
for col in raw.columns:
    s = raw[col].dropna()
    if len(s):
        print(f'{col:22s} | {len(s):5d} months | {s.index.min().date()} → {s.index.max().date()}')
    else:
        print(f'{col:22s} | empty')

## 5. Variable Groups, Sample Window and Cleaning

`categories` is the single source of truth for both the 7 factor groups (macro/bond/fx/commodity/stock/sentiment/uncertainty)
and the 12 proxy-variable categories used in `PVRFClassifier.ipynb`.

In [ ]:
categories = {
    'Production':       ['NEWORDER', 'INDPRO', 'CUMFNS', 'BOPGTB', 'ISRATIO'],
    'Consumer':         ['RPI', 'DPCERA3M086SBEA', 'PCEPILFE', 'RSXFS'],
    'Labour':           ['PAYEMS', 'JTSJOL', 'ICSA', 'CCSA/ICSA', 'UNRATE', 'AHETPI'],
    'Housing':          ['MORTGAGE30US', 'PERMIT', 'COMPUTSA/PERMIT'],
    'Liquidity_Credit': ['BUSLOANS', 'M2SL', 'FEDFUNDS'],
    'Inflation':        ['CPIAUCSL', 'PPIACO', 'MICH', 'EXPINF1YR'],
    'Bond_Market':      ['TB3MS', 'TB6MS', 'GS1', 'GS2', 'GS5', 'GS10', 'BAMLH0A0HYM2', 'T10Y2YM', 'T10Y3MM'],
    'FX_Market':        ['Extended_TWEXBGSMTH', '1/EXSZUS', '1/EXJPUS', 'EXUSUK', 'ECU_EUR_USD', '1/EXCAUS'],
    'Commodity':        ['WPU012202', 'WPU0121', 'GOLD', 'WTISPLC', 'WPUSI019011', 'PNGASUSUSDM'],
    'Stock_Market':     ['^GSPC', '^IXIC', '^DJI', '^N225', '^FCHI', '^AEX', '^BFX', '^GSPTSE', '^HSI', '^FTSE', '^GDAXI'],
    'Sentiment':        ['USACSCICP02STSAM', 'UMCSENT', 'AAII_Bullish', 'AAII_Bearish', 'KCFSI', 'STLFSI4', 'NFCI'],
    'Uncertainty':      ['GPR', 'GPRT', 'GPRA', 'USEPUINDXD', 'EPUMONETARY', 'EPUTRADE', 'VIXCLS'],
}
group_of_category = {
    'Production': 'macro', 'Consumer': 'macro', 'Labour': 'macro', 'Housing': 'macro', 'Liquidity_Credit': 'macro', 'Inflation': 'macro',
    'Bond_Market': 'bond', 'FX_Market': 'fx', 'Commodity': 'commodity', 'Stock_Market': 'stock', 'Sentiment': 'sentiment', 'Uncertainty': 'uncertainty',
}

# Series that are levels and get a year-over-year % change; everything else (rates, spreads, ratios, indices) is used as is
yoy_cols = ['NEWORDER', 'INDPRO', 'BOPGTB', 'RPI', 'DPCERA3M086SBEA', 'PCEPILFE', 'RSXFS', 'BUSLOANS', 'M2SL', 'PAYEMS', 'UNRATE',
            'JTSJOL', 'ICSA', 'AHETPI', 'MORTGAGE30US', 'PERMIT', 'CPIAUCSL', 'PPIACO',
            'USACSCICP02STSAM', 'UMCSENT', 'GPR', 'GPRT', 'GPRA', 'USEPUINDXD', 'EPUMONETARY', 'EPUTRADE']
yoy_cols += categories['FX_Market'] + categories['Commodity'] + categories['Stock_Market']

# Keep only the variables we actually have, and build the column table
rows = []
for cat, cols in categories.items():
    for col in cols:
        if col in raw.columns and raw[col].notna().any():
            rows.append({'variable': col, 'group': group_of_category[cat], 'pv_category': cat,
                         'transform': 'yoy' if col in yoy_cols else 'level'})
        else:
            print(f'Not available, removed: {col}')
columns_df = pd.DataFrame(rows)
all_cols = columns_df['variable'].tolist()
print(f'\n{len(all_cols)} variables across {columns_df["pv_category"].nunique()} categories')

In [ ]:
# Restrict to the modelling window, forward-fill gaps, keep the recession indicator separately
data = raw.loc[start_date:end_date, all_cols].ffill().dropna(how='all')
if 'USREC' not in raw.columns:
    print('WARNING: USREC not available — recession indicator set to 0 everywhere.')
    raw['USREC'] = 0.0
recession = raw.loc[start_date:end_date, ['USREC']].ffill().fillna(0)

print(f'Sample: {data.shape[0]} months × {data.shape[1]} variables, {data.index.min().date()} → {data.index.max().date()}')
print('\nMissing values per column (non-zero only):')
print(data.isnull().sum()[data.isnull().sum() > 0])

## 6. Year-over-Year Changes and Rolling 5-Year Quantile Standardization
Each value is replaced by its quantile rank within the past `rolling_window` months (backward-looking only).

In [ ]:
data_YoY = pd.DataFrame(index=data.index)
for col in all_cols:
    data_YoY[col] = data[col].pct_change(12) * 100 if col in yoy_cols else data[col]
data_YoY = data_YoY.dropna()
print(f'After YoY transforms: {data_YoY.shape}, {data_YoY.index.min().date()} → {data_YoY.index.max().date()}')

In [ ]:
def rolling_quantile_standardize(series, window):
    """Quantile rank of series[t] within the past `window` observations (including t)."""
    result = pd.Series(index=series.index, dtype=float)
    for i in range(window - 1, len(series)):
        window_data = series.iloc[i - (window - 1):i + 1].dropna()
        if pd.isna(series.iloc[i]) or len(window_data) == 0:
            continue
        result.iloc[i] = (window_data <= series.iloc[i]).sum() / len(window_data)
    return result

print('Computing rolling quantile standardization...')
data_q = pd.DataFrame({col: rolling_quantile_standardize(data_YoY[col], rolling_window) for col in all_cols})
data_q = data_q.ffill().dropna()
print(f'Standardized: {data_q.shape}, {data_q.index.min().date()} → {data_q.index.max().date()}')

In [ ]:
# Check plot of the standardized variables
fig, axes = plt.subplots(len(all_cols), 1, figsize=(12, 2.5 * len(all_cols)))
for ax, col in zip(axes, all_cols):
    ax.plot(data_q.index, data_q[col], lw=0.8)
    ax.axhline(0.5, color='gray', lw=0.7, ls='--')
    ax.set_title(col); ax.set_ylim(-0.05, 1.05); ax.set_ylabel('Quantile')
plt.suptitle('Rolling 5-Year Quantile Standardized Variables', y=1.0)
plt.tight_layout(); plt.show()

## 7. Target: Svensson Zero-Coupon Yields
The Fed's Nelson-Siegel-Svensson parameters (`feds200628.csv`, https://www.federalreserve.gov/data/nominal-yield-curve.htm)
are used to reconstruct zero-coupon yields at 13 maturities. These are the marginal-likelihood target of the tree
and the regression target of the PV-RF SHAP weights.

In [ ]:
path = find_file('feds200628.csv')
svensson = pd.read_csv(path, skiprows=9, index_col=0, parse_dates=True)
svensson = svensson[['BETA0', 'BETA1', 'BETA2', 'BETA3', 'TAU1', 'TAU2']].replace(-999.99, np.nan).dropna()
svensson_monthly = svensson.resample('MS').first().dropna()  # TODO (known issue): first-of-month convention, see Section 2
print(f'Svensson parameters: {svensson_monthly.shape}, {svensson_monthly.index.min().date()} → {svensson_monthly.index.max().date()}')

def svensson_yield(tau_months, beta0, beta1, beta2, beta3, tau1, tau2):
    tau = tau_months / 12.0
    f1 = (1 - np.exp(-tau / tau1)) / (tau / tau1)
    f2 = f1 - np.exp(-tau / tau1)
    f3 = (1 - np.exp(-tau / tau2)) / (tau / tau2) - np.exp(-tau / tau2)
    return beta0 + beta1 * f1 + beta2 * f2 + beta3 * f3

maturities_months = [3, 6, 9, 12, 24, 36, 48, 60, 72, 84, 96, 108, 120]
yields_svensson = pd.DataFrame(index=svensson_monthly.index)
for m in maturities_months:
    p = svensson_monthly
    yields_svensson[f'Y{m}M'] = svensson_yield(m, p.BETA0, p.BETA1, p.BETA2, p.BETA3, p.TAU1, p.TAU2)
yields_svensson = yields_svensson.dropna()
print(f'Reconstructed yields: {yields_svensson.shape}')
yields_svensson.head()

## 8. Align and Save

In [ ]:
common_idx = data_q.index.intersection(yields_svensson.index).sort_values()
data_q = data_q.loc[common_idx]
target = pd.concat([yields_svensson.loc[common_idx], svensson_monthly.loc[common_idx]], axis=1)
data.index.name = data_q.index.name = target.index.name = recession.index.name = 'date'

data.to_csv(out_levels)
data_q.to_csv(out_quantiles)
target.to_csv(out_target)
recession.to_csv(out_recession)
columns_df.to_csv(out_columns, index=False)

print(f'Modelling sample: {len(common_idx)} months, {common_idx.min().date()} → {common_idx.max().date()}')
print(f'  {out_levels}: {data.shape}')
print(f'  {out_quantiles}: {data_q.shape}')
print(f'  {out_target}: {target.shape}')
print(f'  {out_recession}: {recession.shape}')
print(f'  {out_columns}: {columns_df.shape}')